In [1]:
import sys, socket
print(sys.executable)
print(socket.gethostname())

/opt/conda/bin/python
jupyter


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.appName("ChurnAnalytics")
         # .master("spark://spark-master:7077")   # uncomment to run on the cluster
         .getOrCreate())
NN = "hdfs://namenode:8020"

df = (spark.read.option("header", True).option("inferSchema", True)
      .csv(NN + "/data/churn/raw/churn.csv"))
df.printSchema()
print("Records:", df.count(), "| Partitions:", df.rdd.getNumPartitions())
df.show(5, truncate=False)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

Records: 7043 | Partitions: 1
+----------+------+-------------+-------+----------

In [3]:
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()
print("Blank TotalCharges:", df.filter(F.trim(F.col("TotalCharges")) == "").count())
print("Duplicate IDs:", df.count() - df.dropDuplicates(["customerID"]).count())
print("Negative charges:", df.filter(F.col("MonthlyCharges") < 0).count())

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|         0|     0|            0|      0|         0|     0|           0|            0|              0|             0|           0|               0|          0|          0|              0|       0|               0| 

In [4]:
clean = (df
  .withColumn("TotalCharges", F.when(F.trim(F.col("TotalCharges")) == "", 0.0)
                               .otherwise(F.col("TotalCharges").cast("double")))
  .withColumn("churn_flag", F.when(F.col("Churn") == "Yes", 1).otherwise(0))
  .withColumn("tenure_group", F.when(F.col("tenure") <= 12, "0-12")
      .when(F.col("tenure") <= 24, "13-24").when(F.col("tenure") <= 48, "25-48").otherwise("49+"))
  .withColumn("spend_category", F.when(F.col("MonthlyCharges") < 35, "Low")
      .when(F.col("MonthlyCharges") < 70, "Medium").otherwise("High")))
clean.cache()
clean.write.mode("overwrite").parquet(NN + "/data/churn/processed/churn_clean")
clean.printSchema(); print(clean.count())

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)
 |-- churn_flag: integer (nullable = false)
 |-- tenure_group: string (nullable = 

In [5]:
for c in ["Contract", "tenure_group", "InternetService", "PaymentMethod", "spend_category"]:
    r = (clean.groupBy(c).agg(F.count("*").alias("customers"),
          F.round(F.avg("churn_flag")*100, 2).alias("churn_pct"),
          F.round(F.avg("MonthlyCharges"), 2).alias("avg_monthly"))
         .orderBy(F.desc("churn_pct")))
    print("== Churn by", c); r.show()
    r.coalesce(1).write.mode("overwrite").option("header", True)\
     .csv(NN + "/data/churn/results/churn_by_" + c.lower())

== Churn by Contract
+--------------+---------+---------+-----------+
|      Contract|customers|churn_pct|avg_monthly|
+--------------+---------+---------+-----------+
|Month-to-month|     3875|    42.71|       66.4|
|      One year|     1473|    11.27|      65.05|
|      Two year|     1695|     2.83|      60.77|
+--------------+---------+---------+-----------+

== Churn by tenure_group
+------------+---------+---------+-----------+
|tenure_group|customers|churn_pct|avg_monthly|
+------------+---------+---------+-----------+
|        0-12|     2186|    47.44|       56.1|
|       13-24|     1024|    28.71|      61.36|
|       25-48|     1594|    20.39|      65.93|
|         49+|     2239|     9.51|      73.95|
+------------+---------+---------+-----------+

== Churn by InternetService
+---------------+---------+---------+-----------+
|InternetService|customers|churn_pct|avg_monthly|
+---------------+---------+---------+-----------+
|    Fiber optic|     3096|    41.89|       91.5|
|    

In [6]:
x = (clean.groupBy("Contract", "PaymentMethod")
     .agg(F.round(F.avg("churn_flag")*100, 2).alias("churn_pct"), F.count("*").alias("n")))
w = Window.partitionBy("Contract").orderBy(F.desc("churn_pct"))
x.withColumn("rank_in_contract", F.row_number().over(w)).orderBy("Contract", "rank_in_contract").show(20, False)

+--------------+-------------------------+---------+----+----------------+
|Contract      |PaymentMethod            |churn_pct|n   |rank_in_contract|
+--------------+-------------------------+---------+----+----------------+
|Month-to-month|Electronic check         |53.73    |1850|1               |
|Month-to-month|Bank transfer (automatic)|34.13    |589 |2               |
|Month-to-month|Credit card (automatic)  |32.78    |543 |3               |
|Month-to-month|Mailed check             |31.58    |893 |4               |
|One year      |Electronic check         |18.44    |347 |1               |
|One year      |Credit card (automatic)  |10.3     |398 |2               |
|One year      |Bank transfer (automatic)|9.72     |391 |3               |
|One year      |Mailed check             |6.82     |337 |4               |
|Two year      |Electronic check         |7.74     |168 |1               |
|Two year      |Bank transfer (automatic)|3.37     |564 |2               |
|Two year      |Credit ca

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

cat = ["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService",
       "OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV",
       "StreamingMovies","Contract","PaperlessBilling","PaymentMethod"]
num = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

stages = [StringIndexer(inputCol=c, outputCol=c+"_i") for c in cat]
stages += [OneHotEncoder(inputCols=[c+"_i" for c in cat], outputCols=[c+"_v" for c in cat]),
           VectorAssembler(inputCols=[c+"_v" for c in cat] + num, outputCol="features"),
           LogisticRegression(featuresCol="features", labelCol="churn_flag")]

train, test = clean.randomSplit([0.8, 0.2], seed=42)
model = Pipeline(stages=stages).fit(train)
pred = model.transform(test)
print("AUC:", BinaryClassificationEvaluator(labelCol="churn_flag").evaluate(pred))
print("Accuracy:", pred.filter(F.col("prediction") == F.col("churn_flag")).count() / pred.count())

AUC: 0.8428253698520558
Accuracy: 0.8074349442379182


In [8]:
from pyspark.ml.functions import vector_to_array
scored = (model.transform(clean.filter("churn_flag = 0"))
          .withColumn("risk", F.round(vector_to_array("probability")[1], 3)))
top = scored.orderBy(F.desc("risk")).select("customerID", "risk", "Contract", "tenure").limit(20).collect()
for r in top:
    print(f"put 'churn_risk','{r.customerID}','risk:score','{r.risk}'")
    print(f"put 'churn_risk','{r.customerID}','profile:contract','{r.Contract}'")
    print(f"put 'churn_risk','{r.customerID}','profile:tenure','{r.tenure}'")

put 'churn_risk','5150-ITWWB','risk:score','0.837'
put 'churn_risk','5150-ITWWB','profile:contract','Month-to-month'
put 'churn_risk','5150-ITWWB','profile:tenure','3'
put 'churn_risk','6630-UJZMY','risk:score','0.821'
put 'churn_risk','6630-UJZMY','profile:contract','Month-to-month'
put 'churn_risk','6630-UJZMY','profile:tenure','4'
put 'churn_risk','2545-EBUPK','risk:score','0.819'
put 'churn_risk','2545-EBUPK','profile:contract','Month-to-month'
put 'churn_risk','2545-EBUPK','profile:tenure','2'
put 'churn_risk','0021-IKXGC','risk:score','0.815'
put 'churn_risk','0021-IKXGC','profile:contract','Month-to-month'
put 'churn_risk','0021-IKXGC','profile:tenure','1'
put 'churn_risk','6350-XFYGW','risk:score','0.801'
put 'churn_risk','6350-XFYGW','profile:contract','Month-to-month'
put 'churn_risk','6350-XFYGW','profile:tenure','4'
put 'churn_risk','4847-TAJYI','risk:score','0.793'
put 'churn_risk','4847-TAJYI','profile:contract','Month-to-month'
put 'churn_risk','4847-TAJYI','profile:tenu

In [9]:
lines = []
for r in top:
    lines.append(f"put 'churn_risk','{r.customerID}','risk:score','{r.risk}'")
    lines.append(f"put 'churn_risk','{r.customerID}','profile:contract','{r.Contract}'")
    lines.append(f"put 'churn_risk','{r.customerID}','profile:tenure','{r.tenure}'")
lines.append("exit")
open("/home/jovyan/hbase_puts.txt", "w").write("\n".join(lines) + "\n")
print(len(lines), "lines written")

61 lines written
